# Generate Historical Data

Use this notebook to generate canonical CSV datasets containing fundamental market prices (without observation noise) using ABIDES' built-in `SparseMeanRevertingOracle`. These CSVs can be loaded as `HistoricalOracleSettings` in your simulations.

In [ ]:
from io import BytesIO

import pandas as pd
import Path
from IPython.display import Image, display
from matplotlib import pyplot as plt

from rohan.simulation.data.generator import generate_fundamental_csv

### Generate the Baseline Dataset

Configure the parameters below to generate a new CSV. The file will be placed in the `src/rohan/simulation/data/datasets/` directory by default so it can be automatically discovered by the Terminal UI.

In [ ]:
DATASET_DIR = Path("../hist_data")

# Default values for all SparseMeanRevertingOracle inputs used by generate_fundamental_csv.
COMMON_SETTINGS = {
    "symbol": "ABM",
    "r_bar": 100_000,
    "kappa": 1.67e-16,
    "fund_vol": 5e-6,
    "megashock_lambda_a": 2.77778e-18,
    "megashock_mean": 1_000,
    "megashock_var": 50_000,
    "date": "20260130",
    "start_time": "09:30:00",
    "end_time": "17:00:00",
    "step_size_ns": int(5e7),
    "seed": 42,
}

# Scenario-specific overrides. Every oracle parameter is available here for tuning.
SCENARIOS = {
    "baseline": {
        "filename": "baseline_scenario.csv",
        "title": "Baseline",
        "color": "tab:blue",
        "seed": 42,
    },
    "volatility": {
        "filename": "vola_scenario.csv",
        "title": "High volatility",
        "color": "tab:red",
        "kappa": 1.67e-15,
        "fund_vol": 5e-4,
        "megashock_lambda_a": 5e-18,
        "megashock_mean": 2_500,
        "megashock_var": 200_000,
        "seed": 314,
    },
    "reversion_shock": {
        "filename": "reversion_shock_scenario.csv",
        "title": "Fast reversion + large shocks",
        "color": "tab:green",
        "kappa": 2.5e-15,
        "fund_vol": 2e-4,
        "megashock_lambda_a": 1e-17,
        "megashock_mean": 5_000,
        "megashock_var": 350_000,
        "seed": 2718,
    },
}

ORACLE_PARAM_KEYS = [
    "symbol",
    "r_bar",
    "kappa",
    "fund_vol",
    "megashock_lambda_a",
    "megashock_mean",
    "megashock_var",
    "date",
    "start_time",
    "end_time",
    "step_size_ns",
    "seed",
]


def build_output_path(filename: str) -> Path:
    return DATASET_DIR / filename


def build_generator_kwargs(scenario: dict) -> dict:
    merged = {**COMMON_SETTINGS, **scenario}
    return {k: merged[k] for k in ORACLE_PARAM_KEYS}


def generate_dataset(name: str, scenario: dict) -> Path:
    output_path = build_output_path(scenario["filename"])
    generate_fundamental_csv(
        output_path=output_path,
        **build_generator_kwargs(scenario),
    )
    print(f"Generated {name} dataset at {output_path.resolve()}")
    return output_path


dataset_paths = {name: generate_dataset(name, scenario) for name, scenario in SCENARIOS.items()}

In [ ]:
scenario_overview = pd.DataFrame(
    [
        {
            "scenario": name,
            "file": cfg["filename"],
            "seed": {**COMMON_SETTINGS, **cfg}["seed"],
            "kappa": {**COMMON_SETTINGS, **cfg}["kappa"],
            "fund_vol": {**COMMON_SETTINGS, **cfg}["fund_vol"],
            "megashock_lambda_a": {**COMMON_SETTINGS, **cfg}["megashock_lambda_a"],
            "megashock_mean": {**COMMON_SETTINGS, **cfg}["megashock_mean"],
            "megashock_var": {**COMMON_SETTINGS, **cfg}["megashock_var"],
        }
        for name, cfg in SCENARIOS.items()
    ]
)

scenario_overview

In [ ]:
def load_dataset(dataset_path: Path) -> pd.DataFrame:
    df = pd.read_csv(dataset_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df


def plot_all_scenarios(dataset_paths: dict, scenarios: dict) -> None:
    fig, ax = plt.subplots(figsize=(12, 5))

    for name, dataset_path in dataset_paths.items():
        df = load_dataset(dataset_path)
        cfg = scenarios[name]
        ax.plot(
            df["timestamp"],
            df["price_cents"],
            label=cfg.get("title", name),
            color=cfg.get("color"),
            linewidth=1.2,
        )

    ax.set_xlabel("Time")
    ax.set_ylabel("Price (cents)")
    ax.set_title("Historical Fundamental Price Scenarios")
    ax.legend(loc="best")
    ax.grid(alpha=0.25)
    fig.autofmt_xdate()
    fig.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    buf.seek(0)
    display(Image(data=buf.getvalue()))
    plt.close(fig)


plot_all_scenarios(dataset_paths, SCENARIOS)